# Improved Squat Embedding & Comparison Pipeline

Changes from `check_data_not_normalized.ipynb`:
1. **Fixed symmetry feature indices** (was comparing wrong columns)
2. **Velocity computed from smoothed angles** (consistent, less noisy)
3. **Added trunk lean angle** (critical for squat form)
4. **Added squat depth feature** (hip height relative to knee)
5. **Rep segmentation** (compare rep-by-rep instead of full video)
6. **Euclidean distance for DTW** (preserves magnitude differences)
7. **Reference-based normalization** (uses pro stats for both videos)
8. **Named feature indices** for interpretability

### Setup & Joint Indices

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from dtw import dtw

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

from Utils.utils.utils import *

# MediaPipe joint indices
HIP_L = 23
HIP_R = 24
KNEE_L = 25
KNEE_R = 26
ANKLE_L = 27
ANKLE_R = 28
HEEL_L = 29
HEEL_R = 30
TOE_L = 31
TOE_R = 32

SHOULDER_L = 11
SHOULDER_R = 12
ELBOW_L = 13
ELBOW_R = 14
WRIST_L = 15
WRIST_R = 16
PINKY_L = 17
PINKY_R = 18

NOSE = 0

# Feature names for interpretability
ANGLE_NAMES = [
    "left_ankle", "right_ankle",
    "left_knee", "right_knee",
    "left_hip", "right_hip",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "trunk_lean",
]

print("Setup complete.")

### Load Data

In [ ]:
# ============================================================
# VIDEO FILE SELECTION - CHANGE THESE TO SWITCH VIDEOS
# ============================================================
REF_FILE = "front_narrow_landmarks.npy"   # template / pro video
USER_FILE = "back_angle_narrow_landmarks.npy"    # user video to analyze

# ============================================================
# Everything below derives from the two filenames above
# ============================================================
LANDMARKS_DIR = "../MediaPipe_landmarks"
ref_path = os.path.join(LANDMARKS_DIR, REF_FILE)
user_path = os.path.join(LANDMARKS_DIR, USER_FILE)

# Clean names for printing and file output (strip _landmarks.npy)
REF_NAME = REF_FILE.replace("_landmarks.npy", "")
USER_NAME = USER_FILE.replace("_landmarks.npy", "")

ref_landmarks = np.load(ref_path)
user_landmarks = np.load(user_path)

print(f"Template (REF):  {REF_NAME}  →  {ref_landmarks.shape}  (frames, joints, xyz)")
print(f"User:            {USER_NAME}  →  {user_landmarks.shape}")
print("\nTo switch videos, change REF_FILE and USER_FILE above and re-run all cells.")

### Compute Angle Features (2D)

Uses `calculate_angle`.
Also adds trunk lean angle: the angle between the torso vector (pelvis->neck) and vertical.

In [ ]:
def compute_angle_2d(p1, p2, p3):
    """
    Compute angle at p2 using only x,y coordinates (ignoring unreliable z).
    Returns inner angle in degrees.
    """
    # Use only x, y (ignore z)
    a = np.array([p1[0] - p2[0], p1[1] - p2[1]])
    b = np.array([p3[0] - p2[0], p3[1] - p2[1]])
    
    cos_angle = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))


def compute_trunk_lean_2d(landmarks):
    """
    Compute trunk lean angle relative to vertical using 2D (x,y only).
    0 = perfectly upright, higher = more forward lean.
    """
    # Use only x, y coordinates
    pelvis = (landmarks[:, HIP_L, :2] + landmarks[:, HIP_R, :2]) / 2.0
    neck = (landmarks[:, SHOULDER_L, :2] + landmarks[:, SHOULDER_R, :2]) / 2.0
    
    torso_vec = neck - pelvis  # (T, 2)
    # In MediaPipe, y-axis points downward, so "up" is negative y
    vertical = np.array([0, -1])
    
    dot = np.einsum('ij,j->i', torso_vec, vertical)
    norms = np.linalg.norm(torso_vec, axis=1) + 1e-8
    cos_angle = np.clip(dot / norms, -1.0, 1.0)
    
    return np.degrees(np.arccos(cos_angle))  # (T,)


def compute_angle_features_2d(landmarks):
    """
    Compute joint angles using only x,y coordinates (2D).
    This is MORE ACCURATE than 3D because MediaPipe's z-estimate 
    from monocular video is unreliable.
    
    Returns: (T, 13) array -- 12 joint angles + trunk lean
    """
    T = landmarks.shape[0]
    features = []

    for f in range(T):
        lm = landmarks[f]

        left_ankle    = compute_angle_2d(lm[KNEE_L],     lm[ANKLE_L],    lm[TOE_L]) # ankle angle = knee-heel-toe?
        right_ankle   = compute_angle_2d(lm[KNEE_R],     lm[ANKLE_R],    lm[TOE_R])
        left_knee     = compute_angle_2d(lm[HIP_L],      lm[KNEE_L],     lm[ANKLE_L])
        right_knee    = compute_angle_2d(lm[HIP_R],      lm[KNEE_R],     lm[ANKLE_R])
        left_hip      = compute_angle_2d(lm[SHOULDER_L], lm[HIP_L],      lm[KNEE_L])
        right_hip     = compute_angle_2d(lm[SHOULDER_R], lm[HIP_R],      lm[KNEE_R])
        left_shoulder = compute_angle_2d(lm[ELBOW_L],    lm[SHOULDER_L], lm[HIP_L])
        right_shoulder= compute_angle_2d(lm[ELBOW_R],    lm[SHOULDER_R], lm[HIP_R])
        left_elbow    = compute_angle_2d(lm[SHOULDER_L], lm[ELBOW_L],    lm[WRIST_L])
        right_elbow   = compute_angle_2d(lm[SHOULDER_R], lm[ELBOW_R],    lm[WRIST_R])
        left_wrist    = compute_angle_2d(lm[ELBOW_L],    lm[WRIST_L],    lm[PINKY_L])
        right_wrist   = compute_angle_2d(lm[ELBOW_R],    lm[WRIST_R],    lm[PINKY_R])

        features.append([
            left_ankle, right_ankle,
            left_knee, right_knee,
            left_hip, right_hip,
            left_shoulder, right_shoulder,
            left_elbow, right_elbow,
            left_wrist, right_wrist,
        ])

    angles = np.array(features)  # (T, 12)
    trunk_lean = compute_trunk_lean_2d(landmarks)  # (T,)
    angles = np.column_stack([angles, trunk_lean])  # (T, 13)

    return angles


# USE 2D ANGLES (much more accurate than 3D for front-facing video)
print("Computing 2D angle features (ignoring unreliable z-coordinate)...")
ref_angles = compute_angle_features_2d(ref_landmarks)
user_angles = compute_angle_features_2d(user_landmarks)

print(f"Reference angles: {ref_angles.shape}")
print(f"User angles:      {user_angles.shape}")
print(f"Feature order: {ANGLE_NAMES}")

# Quick sanity check
print("\nSanity check (should match 2D values from diagnostic):")
print(f"  Reference left_knee - min: {ref_angles[:, 2].min():.1f}°, max: {ref_angles[:, 2].max():.1f}°")
print(f"  User left_knee      - min: {user_angles[:, 2].min():.1f}°, max: {user_angles[:, 2].max():.1f}°")

### Smooth Angles

In [ ]:
def smooth_angles(angles, window=11, polyorder=3):
    """Savitzky-Golay filter. Adjust window to video FPS if needed."""
    # Ensure window doesn't exceed number of frames
    w = min(window, angles.shape[0])
    if w % 2 == 0:
        w -= 1  # must be odd
    return savgol_filter(angles, window_length=w, polyorder=polyorder, axis=0)

ref_smooth = smooth_angles(ref_angles)
user_smooth = smooth_angles(user_angles)

print(f"Smoothed reference: {ref_smooth.shape}")
print(f"Smoothed user:      {user_smooth.shape}")

### Compute Derived Features (from smoothed data)

- **Velocity**: frame-to-frame angle change (from smoothed angles, not raw)
- **Symmetry**: left-right differences (fixed indices)
- **Squat depth**: hip height relative to knee height

In [ ]:
def compute_squat_depth(landmarks):
    """
    Hip Y minus Knee Y (normalized by pelvis-neck distance).
    Positive = hips above knees. Negative = hips below knees (deep squat).
    """
    pelvis = (landmarks[:, HIP_L, :] + landmarks[:, HIP_R, :]) / 2.0
    neck = (landmarks[:, SHOULDER_L, :] + landmarks[:, SHOULDER_R, :]) / 2.0
    
    # Scale factor: pelvis-to-neck distance
    scale = np.linalg.norm(neck - pelvis, axis=1, keepdims=True) + 1e-8
    
    hip_y = pelvis[:, 1:2]  # y coordinate of pelvis
    knee_y = ((landmarks[:, KNEE_L, 1:2] + landmarks[:, KNEE_R, 1:2]) / 2.0)
    
    # Normalize by torso length. In MediaPipe y increases downward,
    # so hip_y < knee_y means hips are above knees
    depth = (hip_y - knee_y) / scale  # negative = deeper squat
    return depth  # (T, 1)


def build_embedding(smooth_angles, landmarks):
    """
    Build the full embedding from smoothed angles and raw landmarks.
    All derived features use smoothed angles for consistency.
    
    Returns: (T-1, D) embedding array and list of feature names.
    """
    # Velocity from smoothed angles (T-1 frames)
    velocity = np.diff(smooth_angles, axis=0)
    
    # Trim everything to match velocity length (T-1)
    T = velocity.shape[0]
    angles_trimmed = smooth_angles[:T]
    
    # Symmetry features from smoothed angles (fixed indices!)
    # Index 2 = left_knee, 3 = right_knee, 4 = left_hip, 5 = right_hip
    knee_symmetry = (angles_trimmed[:, 2] - angles_trimmed[:, 3]).reshape(-1, 1)
    hip_symmetry = (angles_trimmed[:, 4] - angles_trimmed[:, 5]).reshape(-1, 1)
    
    # Squat depth
    depth = compute_squat_depth(landmarks)[:T]  # (T, 1)
    
    embedding = np.concatenate([
        angles_trimmed,     # 13 smoothed angles
        velocity,           # 13 velocity features
        knee_symmetry,      # 1
        hip_symmetry,       # 1
        depth,              # 1
    ], axis=1)
    
    # Build feature name list
    feature_names = (
        [f"angle_{n}" for n in ANGLE_NAMES] +
        [f"vel_{n}" for n in ANGLE_NAMES] +
        ["knee_symmetry", "hip_symmetry", "squat_depth"]
    )
    
    return embedding, feature_names


ref_embedding, feature_names = build_embedding(ref_smooth, ref_landmarks)
user_embedding, _ = build_embedding(user_smooth, user_landmarks)

print(f"Reference embedding: {ref_embedding.shape}")
print(f"User embedding:      {user_embedding.shape}")
print(f"Total features: {len(feature_names)}")
print(f"Features: {feature_names}")

### Normalize Embeddings

Uses the **reference athlete's** mean/std to normalize both videos.
This preserves real differences in range of motion instead of squashing them.

In [ ]:
# Compute stats from reference (pro) video
ref_mean = ref_embedding.mean(axis=0)
ref_std = ref_embedding.std(axis=0) + 1e-8

# Normalize both using the reference stats
ref_norm = (ref_embedding - ref_mean) / ref_std
user_norm = (user_embedding - ref_mean) / ref_std

print(f"Reference normalized: {ref_norm.shape}")
print(f"User normalized:      {user_norm.shape}")
print(f"\nRef mean after normalization (should be ~0):  {ref_norm.mean(axis=0)[:3].round(4)}...")
print(f"User mean after normalization (deviations from pro): {user_norm.mean(axis=0)[:3].round(4)}...")

### (Optional) Rep Segmentation

Detect individual squat reps by finding minima in knee angle.
This lets you compare rep-by-rep instead of the full video.

**distance=100**: Minimum 100 frames (~3.3 seconds at 30fps) between detected bottoms. This prevents detecting small wobbles within one rep as separate reps.

**prominence=25**: The bottom must be at least 25° lower than the surrounding peaks. This filters out tiny dips that aren't real squats.

In [ ]:
# ============================================================
# EXERCISE CONFIGURATION
# ============================================================
EXERCISE_CONFIGS = {
    'heavy_squat': {
        'min_distance': 100,   # ~3.3s at 30fps - heavy squats are slow
        'prominence': 25,      # large angle change required
        'description': 'Heavy barbell squats (slow, controlled reps)'
    },
    'bodyweight_squat': {
        'min_distance': 45,    # ~1.5s at 30fps - faster reps
        'prominence': 15,      # smaller range of motion possible
        'description': 'Bodyweight or goblet squats (moderate speed)'
    },
    'jump_squat': {
        'min_distance': 25,    # ~0.8s at 30fps - explosive reps
        'prominence': 10,      # quick, shallow dips count
        'description': 'Jump squats or plyometric squats (fast, explosive)'
    },
    'adaptive': {
        'description': 'Auto-detect tempo from the knee angle signal'
        # min_distance and prominence computed dynamically
    }
}

# SELECT YOUR EXERCISE TYPE HERE
EXERCISE_TYPE = 'heavy_squat'  # <-- CHANGE THIS


def estimate_adaptive_params(smooth_angles, fps=30):
    """
    Automatically estimate rep detection parameters from the signal.
    
    Looks at the knee angle signal to estimate:
    - Average rep duration (for min_distance)
    - Average angle drop (for prominence)
    """
    knee_angle = (smooth_angles[:, 2] + smooth_angles[:, 3]) / 2.0
    
    # First pass: find peaks with very loose params to estimate tempo
    bottoms, props = find_peaks(-knee_angle, distance=15, prominence=5)
    
    if len(bottoms) < 2:
        # Can't estimate, use conservative defaults
        print("  Adaptive: Not enough peaks found, using heavy_squat defaults")
        return 100, 25
    
    # Average distance between consecutive bottoms
    avg_distance = np.mean(np.diff(bottoms))
    
    # Set min_distance to 60% of average (allows some variation)
    min_distance = int(avg_distance * 0.6)
    
    # Prominence: use median of detected prominences, scaled down slightly
    median_prominence = np.median(props['prominences'])
    prominence = max(5, median_prominence * 0.5)
    
    print(f"  Adaptive: detected ~{len(bottoms)} potential reps")
    print(f"  Adaptive: avg rep distance = {avg_distance:.0f} frames ({avg_distance/fps:.1f}s)")
    print(f"  Adaptive: min_distance = {min_distance}, prominence = {prominence:.1f}")
    
    return min_distance, prominence


def find_rep_boundaries(smooth_angles, exercise_type='heavy_squat'):
    """
    Find squat rep boundaries by detecting knee angle minima (bottom of squat).
    Uses average of left + right knee for a more stable signal.
    
    Args:
        smooth_angles: (T, D) smoothed angle features
        exercise_type: one of 'heavy_squat', 'bodyweight_squat', 'jump_squat', 'adaptive'
    
    Returns list of (start, end) frame indices for each rep.
    """
    config = EXERCISE_CONFIGS[exercise_type]
    print(f"  Exercise type: {exercise_type} - {config['description']}")
    
    if exercise_type == 'adaptive':
        min_distance, prominence = estimate_adaptive_params(smooth_angles)
    else:
        min_distance = config['min_distance']
        prominence = config['prominence']
    
    print(f"  Parameters: min_distance={min_distance}, prominence={prominence}")
    
    # Average left knee (col 2) and right knee (col 3) for stability
    knee_angle = (smooth_angles[:, 2] + smooth_angles[:, 3]) / 2.0
    
    # Find bottoms (minima) = peaks in inverted signal
    bottoms, properties = find_peaks(-knee_angle, distance=min_distance, prominence=prominence)
    
    print(f"  Detected {len(bottoms)} squat bottoms at frames: {bottoms}")
    print(f"  Knee angles at bottoms: {[f'{knee_angle[b]:.1f}' for b in bottoms]}")
    
    if len(bottoms) < 1:
        print("  WARNING: Could not detect any reps. Using full video.")
        return [(0, len(knee_angle) - 1)], knee_angle
    
    # Split at midpoints between consecutive bottoms
    boundaries = [0]
    for i in range(len(bottoms) - 1):
        mid = (bottoms[i] + bottoms[i+1]) // 2
        boundaries.append(mid)
    boundaries.append(len(knee_angle) - 1)
    
    reps = []
    for i in range(len(boundaries) - 1):
        reps.append((boundaries[i], boundaries[i+1]))
    
    return reps, knee_angle


print("Reference video:")
ref_reps, ref_knee = find_rep_boundaries(ref_smooth, exercise_type=EXERCISE_TYPE)
print(f"  Reps detected: {len(ref_reps)}")
for i, (s, e) in enumerate(ref_reps):
    print(f"    Rep {i+1}: frames {s}-{e} ({e-s} frames)")

print("\nUser video:")
user_reps, user_knee = find_rep_boundaries(user_smooth, exercise_type=EXERCISE_TYPE)
print(f"  Reps detected: {len(user_reps)}")
for i, (s, e) in enumerate(user_reps):
    print(f"    Rep {i+1}: frames {s}-{e} ({e-s} frames)")

# Plot knee angle signals so you can visually verify rep detection
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)

axes[0].plot(ref_knee, label="Avg knee angle")
for i, (s, e) in enumerate(ref_reps):
    axes[0].axvline(s, color='r', linestyle='--', alpha=0.5)
axes[0].set_title(f"Reference - {len(ref_reps)} reps detected ({EXERCISE_TYPE})")
axes[0].set_ylabel("Knee angle (degrees)")
axes[0].legend()

axes[1].plot(user_knee, label="Avg knee angle")
for i, (s, e) in enumerate(user_reps):
    axes[1].axvline(s, color='r', linestyle='--', alpha=0.5)
axes[1].set_title(f"User - {len(user_reps)} reps detected ({EXERCISE_TYPE})")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Knee angle (degrees)")
axes[1].legend()

plt.tight_layout()
plt.show()

### Per-Feature DTW Comparison

Instead of one DTW across all 29 features (where Euclidean distance explodes 
due to curse of dimensionality), we run DTW on each angle independently.

After DTW alignment, we compute **Pearson correlation** between the aligned 
signals. This gives a score in [-1, 1] where:
- **1.0** = identical movement pattern
- **0.0** = unrelated
- **-1.0** = opposite movement

This approach is general -- it works for any exercise without custom rules.

In [ ]:
def per_feature_dtw_similarity(ref_angles, user_angles, feature_names):
    """
    Run DTW independently on each feature (1D signal).
    After alignment, compute Pearson correlation and mean absolute error.
    
    Returns dict with per-feature results and overall similarity.
    """
    n_features = ref_angles.shape[1]
    results = []
    
    for i in range(n_features):
        ref_signal = ref_angles[:, i]
        user_signal = user_angles[:, i]
        
        # DTW on 1D signals (absolute difference as distance)
        d, c, a, p = dtw(
            ref_signal.reshape(-1, 1),
            user_signal.reshape(-1, 1),
            dist=lambda x, y: abs(x[0] - y[0])
        )
        
        idx_r, idx_u = p
        
        # Aligned signals
        ref_aligned = ref_signal[idx_r]
        user_aligned = user_signal[idx_u]
        
        # Pearson correlation (pattern similarity)
        correlation = np.corrcoef(ref_aligned, user_aligned)[0, 1]
        
        # Mean absolute error in degrees (magnitude difference)
        mae_degrees = np.mean(np.abs(ref_aligned - user_aligned))
        
        results.append({
            'feature': feature_names[i],
            'correlation': correlation,
            'mae_degrees': mae_degrees,
        })
    
    return results


# Run per-feature comparison on smoothed angles (not the full embedding)
results = per_feature_dtw_similarity(ref_smooth, user_smooth, ANGLE_NAMES)

print(f"{'Feature':<20} {'Correlation':>12} {'MAE (degrees)':>14}")
print("-" * 48)
for r in results:
    print(f"{r['feature']:<20} {r['correlation']:>12.4f} {r['mae_degrees']:>12.1f}")

# Overall similarity: average correlation across all features
correlations = [r['correlation'] for r in results]
overall_similarity = np.mean(correlations)
print(f"\n{'Overall similarity (mean correlation):':<40} {overall_similarity:.4f}")

# Core squat features only (knees, hips, trunk lean)
core_features = ['left_knee', 'right_knee', 'left_hip', 'right_hip', 'trunk_lean']
core_corrs = [r['correlation'] for r in results if r['feature'] in core_features]
core_similarity = np.mean(core_corrs)
print(f"{'Core squat similarity (knee/hip/trunk):':<40} {core_similarity:.4f}")

### Per-Rep Comparison

Compare matching reps using the same per-feature DTW approach.

In [ ]:
n_reps_to_compare = min(len(ref_reps), len(user_reps))

core_features = ['left_knee', 'right_knee', 'left_hip', 'right_hip', 'trunk_lean']

for i in range(n_reps_to_compare):
    rs, re = ref_reps[i]
    us, ue = user_reps[i]
    
    ref_rep_angles = ref_smooth[rs:re+1]
    user_rep_angles = user_smooth[us:ue+1]
    
    if len(ref_rep_angles) < 5 or len(user_rep_angles) < 5:
        print(f"Rep {i+1}: too short, skipping")
        continue
    
    rep_results = per_feature_dtw_similarity(ref_rep_angles, user_rep_angles, ANGLE_NAMES)
    
    all_corrs = [r['correlation'] for r in rep_results]
    core_corrs = [r['correlation'] for r in rep_results if r['feature'] in core_features]
    
    print(f"Rep {i+1}:")
    print(f"  Overall similarity:    {np.mean(all_corrs):.4f}")
    print(f"  Core squat similarity: {np.mean(core_corrs):.4f}")
    for r in rep_results:
        marker = " ***" if r['feature'] in core_features else ""
        print(f"    {r['feature']:<20} corr={r['correlation']:>7.4f}  MAE={r['mae_degrees']:>6.1f} deg{marker}")
    print()

### Visualize Per-Feature Alignment

Plot the DTW-aligned signals for the core squat features to see where the user deviates from the reference.

In [ ]:
core_indices = [i for i, n in enumerate(ANGLE_NAMES) if n in core_features]

fig, axes = plt.subplots(len(core_indices), 1, figsize=(14, 3 * len(core_indices)), sharex=True)

for ax_idx, feat_idx in enumerate(core_indices):
    ref_signal = ref_smooth[:, feat_idx]
    user_signal = user_smooth[:, feat_idx]
    
    # DTW alignment
    d, c, a, p = dtw(
        ref_signal.reshape(-1, 1),
        user_signal.reshape(-1, 1),
        dist=lambda x, y: abs(x[0] - y[0])
    )
    
    ref_aligned = ref_signal[p[0]]
    user_aligned = user_signal[p[1]]
    corr = np.corrcoef(ref_aligned, user_aligned)[0, 1]
    
    axes[ax_idx].plot(ref_aligned, label='Reference', alpha=0.8)
    axes[ax_idx].plot(user_aligned, label='User', alpha=0.8)
    axes[ax_idx].fill_between(
        range(len(ref_aligned)),
        ref_aligned, user_aligned,
        alpha=0.15, color='red'
    )
    axes[ax_idx].set_ylabel('Degrees')
    axes[ax_idx].set_title(f"{ANGLE_NAMES[feat_idx]}  (correlation: {corr:.4f})")
    axes[ax_idx].legend(loc='upper right')

axes[-1].set_xlabel('DTW-aligned frame index')
plt.tight_layout()
plt.show()

### Per-Feature Error Summary

Which joints have the largest absolute deviation in degrees?

In [ ]:
# Sort by MAE (largest deviation first)
sorted_results = sorted(results, key=lambda r: r['mae_degrees'], reverse=True)

print(f"{'Feature':<20} {'Corr':>8} {'MAE (deg)':>10} {'Assessment'}")
print("-" * 60)
for r in sorted_results:
    corr = r['correlation']
    if corr > 0.9:
        assessment = "Excellent"
    elif corr > 0.7:
        assessment = "Good"
    elif corr > 0.5:
        assessment = "Moderate"
    elif corr > 0.3:
        assessment = "Poor"
    else:
        assessment = "Very different"
    
    marker = " <--" if r['feature'] in core_features else ""
    print(f"{r['feature']:<20} {corr:>8.4f} {r['mae_degrees']:>8.1f}   {assessment}{marker}")

### Save Embeddings

In [ ]:
os.makedirs("../embedding", exist_ok=True)

# File names include video names so different comparisons don't overwrite each other
ref_emb_path = f"../embedding/{REF_NAME}_embedding.npy"
user_emb_path = f"../embedding/{USER_NAME}_embedding.npy"
stats_path = f"../embedding/{REF_NAME}_normalization_stats.npz"

np.save(ref_emb_path, ref_embedding)
np.save(user_emb_path, user_embedding)
np.savez(stats_path, mean=ref_mean, std=ref_std)

print(f"Saved: {ref_emb_path}")
print(f"Saved: {user_emb_path}")
print(f"Saved: {stats_path}")


### Why per-feature DTW works better

| Problem with multi-dim DTW | Per-feature DTW solution |
|---|---|
| Euclidean distance in 29D explodes (curse of dimensionality) | Each DTW operates on a 1D signal |
| Noisy features (wrists, elbows) dominate the distance | Each feature gets its own similarity score |
| `exp(-dist)` always near zero in high dimensions | Pearson correlation gives intuitive [-1, 1] range |
| Can't tell *which* joint is the problem | Per-feature breakdown shows exactly where user deviates |
| Single number hides everything | Core feature average focuses on what matters for the exercise |

This approach is **exercise-agnostic**: just change which features are in `core_features` for different exercises (e.g., add shoulders for overhead press).

---

## Rep-to-Template Comparison (Production Approach)

For real-world use, users upload videos with varying lengths and rep counts. 
We can't assume matching rep counts between user and pro videos.

**Solution:** Create a single "template rep" from the pro video by averaging all 
pro reps together. Then compare every user rep against this one template.

Steps:
1. Segment pro video into individual reps
2. DTW-align all pro reps to the same length, then average them into a template
3. For any user video: segment reps, compare each rep to the template
4. Score each rep independently

In [ ]:
def extract_rep_angles(smooth_angles, reps):
    """Extract angle data for each rep as a list of arrays."""
    rep_data = []
    for start, end in reps:
        rep_angles = smooth_angles[start:end+1]
        if len(rep_angles) >= 5:  # minimum viable rep length
            rep_data.append(rep_angles)
    return rep_data


def resample_to_length(signal, target_length):
    """
    Resample a 1D or 2D signal to a target length using linear interpolation.
    signal: (T,) or (T, D)
    
    Only needed for:
    - 'average' template method (must match lengths before averaging)
    - Visualization (plotting signals of different lengths side by side)
    NOT needed before DTW (DTW handles different lengths natively).
    """
    if signal.ndim == 1:
        signal = signal.reshape(-1, 1)
    
    T, D = signal.shape
    x_old = np.linspace(0, 1, T)
    x_new = np.linspace(0, 1, target_length)
    
    resampled = np.zeros((target_length, D))
    for d in range(D):
        resampled[:, d] = np.interp(x_new, x_old, signal[:, d])
    
    return resampled.squeeze() if D == 1 else resampled


def score_rep_quality(rep_angles, core_feature_indices):
    """
    Only used if TEMPLATE_METHOD = 'best'.
    Score a rep's quality based on smoothness and range of motion.
    
    Higher score = better rep (smoother movement, good range of motion)
    """
    # Smoothness: lower velocity variance = smoother movement
    velocity = np.diff(rep_angles, axis=0)
    smoothness = -np.mean(np.var(velocity[:, core_feature_indices], axis=0))
    
    # Range of motion: larger range on core features = deeper squat
    rom = np.mean(np.ptp(rep_angles[:, core_feature_indices], axis=0))
    
    # Combined score (weighted)
    score = smoothness * 0.3 + rom * 0.7
    return score


def create_template_rep(pro_reps_angles, target_length=100, method='average', 
                        core_feature_indices=None):
    """
    Create a template rep from pro video reps.
    
    For 'first' and 'best': returns the raw rep data (no resampling).
    For 'average': resamples all reps to target_length, then averages.
    
    Args:
        pro_reps_angles: list of (T_i, D) arrays, one per rep
        target_length: number of frames in the template (only used for 'average')
        method: 'average', 'first', or 'best'
        core_feature_indices: indices of core features (for 'best' method)
    
    Returns:
        template: (T, D) template rep
        info: dict with metadata
    """
    if len(pro_reps_angles) == 0:
        raise ValueError("No pro reps provided")
    
    n_features = pro_reps_angles[0].shape[1]
    info = {'method': method, 'n_reps_available': len(pro_reps_angles)}
    
    if method == 'first':
        # Use first rep directly - no resampling needed
        template = pro_reps_angles[0]
        info['rep_used'] = 1
        print(f"Template created from FIRST rep (rep #1, {template.shape[0]} frames)")
        
    elif method == 'best':
        # Find the best rep, use it directly
        if core_feature_indices is None:
            core_feature_indices = list(range(n_features))
        
        scores = []
        for i, rep in enumerate(pro_reps_angles):
            score = score_rep_quality(rep, core_feature_indices)
            scores.append(score)
            print(f"  Rep {i+1} quality score: {score:.4f}")
        
        best_idx = np.argmax(scores)
        template = pro_reps_angles[best_idx]
        info['rep_used'] = best_idx + 1
        info['rep_scores'] = scores
        print(f"Template created from BEST rep (rep #{best_idx + 1}, {template.shape[0]} frames)")
        
    elif method == 'average':
        # Resample all reps to same length (REQUIRED for averaging), then average
        resampled_reps = []
        for rep in pro_reps_angles:
            resampled = resample_to_length(rep, target_length)
            resampled_reps.append(resampled)
        
        stacked = np.stack(resampled_reps, axis=0)  # (n_reps, target_length, D)
        template = np.mean(stacked, axis=0)  # (target_length, D)
        info['n_reps_averaged'] = len(pro_reps_angles)
        print(f"Template created by AVERAGING {len(pro_reps_angles)} reps (resampled to {target_length} frames)")
        
    else:
        raise ValueError(f"Unknown method: {method}. Use 'average', 'first', or 'best'")
    
    print(f"Template shape: {template.shape}")
    return template, info


# Extract pro reps
pro_reps_data = extract_rep_angles(ref_smooth, ref_reps)
print(f"Extracted {len(pro_reps_data)} pro reps")
for i, rep in enumerate(pro_reps_data):
    print(f"  Pro rep {i+1}: {rep.shape[0]} frames")
print()

# ============================================================
# TEMPLATE METHOD SELECTION
# Change this to 'average', 'first', or 'best'
# ============================================================
TEMPLATE_METHOD = 'first'  # <-- CHANGE THIS TO SWITCH METHODS
TEMPLATE_LENGTH = 100      # Only used for 'average' method

# Get core feature indices for 'best' method scoring
core_indices = [i for i, n in enumerate(ANGLE_NAMES) if n in core_features]

# Create template
print(f"\nCreating template using method: '{TEMPLATE_METHOD}'")
print("-" * 50)
template_rep, template_info = create_template_rep(
    pro_reps_data, 
    target_length=TEMPLATE_LENGTH, 
    method=TEMPLATE_METHOD,
    core_feature_indices=core_indices
)
print(f"\nTemplate info: {template_info}")

### Visualize Template Rep

Show the averaged template alongside individual pro reps for the core features.

In [ ]:
# Visualize template for core features
core_indices = [i for i, n in enumerate(ANGLE_NAMES) if n in core_features]

fig, axes = plt.subplots(len(core_indices), 1, figsize=(12, 3 * len(core_indices)), sharex=True)

for ax_idx, feat_idx in enumerate(core_indices):
    # Plot individual pro reps (resampled)
    for i, rep in enumerate(pro_reps_data):
        resampled = resample_to_length(rep[:, feat_idx], TEMPLATE_LENGTH)
        axes[ax_idx].plot(resampled, alpha=0.3, color='blue', label='Pro reps' if i == 0 else '')
    
    # Plot template (thick black line)
    axes[ax_idx].plot(template_rep[:, feat_idx], color='black', linewidth=2.5, label='Template')
    
    axes[ax_idx].set_ylabel('Degrees')
    axes[ax_idx].set_title(f"{ANGLE_NAMES[feat_idx]}")
    axes[ax_idx].legend(loc='upper right')

axes[-1].set_xlabel('Normalized frame (0-100)')
plt.suptitle('Pro Template Rep (averaged from all pro reps)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Compare User Reps to Template

Now compare each user rep against the single pro template. This works regardless 
of how many reps the user performs.

In [ ]:
def inner_to_flexion(inner_angle):
    """
    Convert MediaPipe inner knee angle to flexion angle.
    - Inner angle: ~180° when standing, ~90° at parallel
    - Flexion angle: 0° when standing, 90° at parallel, >90° below parallel
    """
    return 180.0 - inner_angle


def compare_rep_to_template(user_rep_angles, template, feature_names, core_features):
    """
    Compare a single user rep to the pro template using DTW.
    
    DTW handles different lengths natively - no resampling needed.
    
    Scoring:
    - Knee angles: penalizes shallower depth, rewards matching or exceeding
    - Other joints: penalizes MAE via exponential decay
    - Both combined with pattern correlation
    """
    results = []
    
    for i, name in enumerate(feature_names):
        template_signal = template[:, i]
        user_signal = user_rep_angles[:, i]
        
        # DTW alignment (handles different lengths natively)
        d, cost_matrix, acc_cost, path = dtw(
            template_signal.reshape(-1, 1),
            user_signal.reshape(-1, 1),
            dist=lambda x, y: abs(x[0] - y[0])
        )
        
        idx_template, idx_user = path
        template_aligned = template_signal[idx_template]
        user_aligned = user_signal[idx_user]
        
        # Pattern correlation
        correlation = np.corrcoef(template_aligned, user_aligned)[0, 1]
        if np.isnan(correlation):
            correlation = 0
        
        # Mean absolute error
        mae_degrees = np.mean(np.abs(template_aligned - user_aligned))
        
        # For KNEE angles: penalize shallower, reward deeper
        if 'knee' in name:
            template_min = template_signal.min()
            user_min = user_signal.min()
            
            if user_min <= template_min:
                depth_penalty = 1.0
            else:
                depth_shortfall = user_min - template_min
                depth_penalty = np.exp(-depth_shortfall / 20.0)
            
            combined_score = correlation * depth_penalty
        else:
            mae_penalty = np.exp(-mae_degrees / 30.0)
            combined_score = correlation * mae_penalty
            depth_penalty = mae_penalty
        
        results.append({
            'feature': name,
            'correlation': correlation,
            'mae_degrees': mae_degrees,
            'penalty': depth_penalty,
            'combined_score': combined_score,
            'is_core': name in core_features
        })
    
    # Summary scores
    all_combined = [r['combined_score'] for r in results if not np.isnan(r['combined_score'])]
    core_combined = [r['combined_score'] for r in results if r['is_core'] and not np.isnan(r['combined_score'])]
    
    # Depth analysis
    template_knee_inner = (template[:, 2].min() + template[:, 3].min()) / 2
    user_knee_inner = (user_rep_angles[:, 2].min() + user_rep_angles[:, 3].min()) / 2
    
    template_flexion = inner_to_flexion(template_knee_inner)
    user_flexion = inner_to_flexion(user_knee_inner)
    
    hit_parallel = user_knee_inner <= 90.0
    
    if user_knee_inner <= template_knee_inner:
        depth_score = 100.0
    else:
        shortfall = user_knee_inner - template_knee_inner
        depth_score = max(0, 100 - shortfall * 2)
    
    # Resample for visualization only
    user_resampled_for_viz = resample_to_length(user_rep_angles, template.shape[0])
    
    return {
        'per_feature': results,
        'overall_similarity': np.mean(all_combined) if all_combined else 0,
        'core_similarity': np.mean(core_combined) if core_combined else 0,
        'user_resampled': user_resampled_for_viz,
        'template_flexion': template_flexion,
        'user_flexion': user_flexion,
        'template_inner': template_knee_inner,
        'user_inner': user_knee_inner,
        'hit_parallel': hit_parallel,
        'depth_score': depth_score
    }


# Extract user reps
user_reps_data = extract_rep_angles(user_smooth, user_reps)
print(f"User video has {len(user_reps_data)} reps\n")

# Compare each user rep to the template
user_rep_scores = []

template_inner = (template_rep[:, 2].min() + template_rep[:, 3].min()) / 2
template_flexion = inner_to_flexion(template_inner)
print(f"Template depth: {template_flexion:.1f}° flexion (inner: {template_inner:.1f}°)")
print(f"Template length: {template_rep.shape[0]} frames (raw, no resampling)")
print("Parallel = 90° flexion. Below parallel = >90° flexion.\n")

for i, user_rep in enumerate(user_reps_data):
    print(f"{'='*70}")
    print(f"USER REP {i+1} ({user_rep.shape[0]} frames) vs PRO TEMPLATE ({template_rep.shape[0]} frames)")
    print(f"{'='*70}")
    
    result = compare_rep_to_template(user_rep, template_rep, ANGLE_NAMES, core_features)
    user_rep_scores.append(result)
    
    depth_status = "GOOD" if result['user_flexion'] >= template_flexion else "SHALLOWER"
    parallel_status = "Hit parallel" if result['hit_parallel'] else "DID NOT HIT PARALLEL"
    
    print(f"User depth: {result['user_flexion']:.1f}° flexion (template: {template_flexion:.1f}°) {depth_status}")
    print(f"Depth check: {parallel_status}")
    print(f"Depth score: {result['depth_score']:.0f}/100")
    print()
    print(f"Core squat similarity:  {result['core_similarity']:.2%}")
    print()
    print(f"{'Feature':<16} {'Corr':>6} {'MAE':>7} {'Penalty':>8} {'Score':>7} {'Grade'}")
    print("-" * 60)
    
    for r in result['per_feature']:
        score = r['combined_score']
        if np.isnan(score):
            grade = "N/A"
        elif score > 0.80:
            grade = "A"
        elif score > 0.60:
            grade = "B"
        elif score > 0.40:
            grade = "C"
        elif score > 0.20:
            grade = "D"
        else:
            grade = "F"
        
        marker = " ***" if r['is_core'] else ""
        print(f"{r['feature']:<16} {r['correlation']:>6.2f} {r['mae_degrees']:>6.1f}° {r['penalty']:>7.2f} {score:>7.2%}  {grade}{marker}")
    print()

# Summary
print("="*70)
print("SUMMARY: Rep Rankings (Combined Score = 60% Similarity + 40% Depth)")
print("="*70)

print(f"\n{'Rep':<6} {'Similarity':<12} {'Depth':<14} {'Depth Score':<12} {'Combined':<10} {'Rank'}")
print("-" * 70)

combined_scores = []
for i, r in enumerate(user_rep_scores):
    combined = r['core_similarity'] * 0.6 + (r['depth_score'] / 100) * 0.4
    combined_scores.append(combined)

ranking = np.argsort(combined_scores)[::-1] + 1

for i in range(len(user_rep_scores)):
    r = user_rep_scores[i]
    rank = list(ranking).index(i+1) + 1
    depth_str = f"{r['user_flexion']:.1f}°"
    if not r['hit_parallel']:
        depth_str += " X"
    print(f"Rep {i+1:<3} {r['core_similarity']:>8.1%}     {depth_str:<14} {r['depth_score']:>6.0f}/100     {combined_scores[i]:>7.1%}     #{rank}")

print(f"\nRanking: {list(ranking)}")

### Visualize User Reps vs Template

Show each user rep overlaid on the pro template for the core features.

In [ ]:
n_user_reps = len(user_rep_scores)
n_core = len(core_indices)

fig, axes = plt.subplots(n_core, n_user_reps, figsize=(5 * n_user_reps, 3 * n_core), squeeze=False)

for rep_idx, rep_result in enumerate(user_rep_scores):
    user_resampled = rep_result['user_resampled']
    core_score = rep_result['core_similarity']
    
    for feat_row, feat_idx in enumerate(core_indices):
        ax = axes[feat_row, rep_idx]
        
        template_signal = template_rep[:, feat_idx]
        user_signal = user_resampled[:, feat_idx]
        
        # Find per-feature correlation
        corr = np.corrcoef(template_signal, user_signal)[0, 1]
        
        ax.plot(template_signal, 'k-', linewidth=2, label='Pro template')
        ax.plot(user_signal, 'r-', linewidth=1.5, alpha=0.8, label='User')
        ax.fill_between(range(len(template_signal)), template_signal, user_signal,
                       alpha=0.2, color='red')
        
        ax.set_title(f"{ANGLE_NAMES[feat_idx]} (r={corr:.3f})", fontsize=10)
        
        if feat_row == 0:
            ax.set_title(f"Rep {rep_idx+1} - {ANGLE_NAMES[feat_idx]} (r={corr:.3f})", fontsize=10)
        
        if rep_idx == 0:
            ax.set_ylabel('Degrees')
        if feat_row == n_core - 1:
            ax.set_xlabel('Normalized frame')
        if feat_row == 0 and rep_idx == n_user_reps - 1:
            ax.legend(loc='upper right', fontsize=8)

plt.suptitle('User Reps vs Pro Template (Core Features)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Generate Actionable Feedback

Based on the comparison, generate specific feedback for the user.

In [ ]:
def generate_feedback(rep_result, rep_number):
    """
    Generate actionable feedback based on rep comparison results.
    """
    feedback = []
    feedback.append(f"\n📊 REP {rep_number} ANALYSIS")
    feedback.append("=" * 40)
    
    core_score = rep_result['core_similarity']
    
    # Overall grade
    if core_score > 0.95:
        overall_grade = "Excellent! 🌟"
    elif core_score > 0.90:
        overall_grade = "Great form! ✅"
    elif core_score > 0.80:
        overall_grade = "Good, minor adjustments needed"
    elif core_score > 0.70:
        overall_grade = "Needs improvement"
    else:
        overall_grade = "Significant form issues"
    
    feedback.append(f"Overall: {overall_grade} (score: {core_score:.2%})")
    feedback.append("")
    
    # Find problem areas (core features with low correlation or high MAE)
    problems = []
    for r in rep_result['per_feature']:
        if not r['is_core']:
            continue
        
        corr = r['correlation']
        mae = r['mae_degrees']
        feature = r['feature']
        
        if corr < 0.85 or mae > 10:
            problems.append({
                'feature': feature,
                'correlation': corr,
                'mae': mae
            })
    
    if problems:
        feedback.append("⚠️  Areas to improve:")
        for p in sorted(problems, key=lambda x: x['correlation']):
            feat = p['feature']
            
            # Generate specific advice based on feature
            if 'knee' in feat:
                if p['mae'] > 15:
                    advice = "Knee angle differs significantly - check squat depth"
                else:
                    advice = "Knee tracking slightly off - focus on knee-over-toe alignment"
            elif 'hip' in feat:
                advice = "Hip hinge pattern differs - practice hip mobility"
            elif 'trunk_lean' in feat:
                advice = "Trunk angle differs - focus on keeping chest up"
            else:
                advice = "Movement pattern differs from pro"
            
            feedback.append(f"   • {feat}: {advice} (MAE: {p['mae']:.1f}°)")
    else:
        feedback.append("✅ All core movements match the pro template well!")
    
    return "\n".join(feedback)


# Generate feedback for each rep
print("\n" + "="*60)
print("FEEDBACK REPORT")
print("="*60)

for i, result in enumerate(user_rep_scores):
    print(generate_feedback(result, i + 1))

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
avg_core = np.mean([r['core_similarity'] for r in user_rep_scores])
print(f"Average core similarity across all reps: {avg_core:.2%}")

best_rep = np.argmax([r['core_similarity'] for r in user_rep_scores]) + 1
worst_rep = np.argmin([r['core_similarity'] for r in user_rep_scores]) + 1
print(f"Best rep: #{best_rep}")
print(f"Rep needing most work: #{worst_rep}")

### Save Template for Production Use

Save the template so it can be loaded in the production app without re-processing the pro video.

### Diagnostic: Understanding the Rep Differences

Let's analyze WHY Rep 2 scores highest. The large MAE on Rep 1 (34° vs 14° for Rep 2) 
suggests the movements are fundamentally different. Let's check:

In [ ]:
# DIAGNOSTIC: Analyze the actual angle ranges to understand the differences

print("="*70)
print("DIAGNOSTIC: Comparing Squat Depth & Angle Ranges")
print("="*70)

# Template (PRO) statistics
print("\n📊 PRO TEMPLATE (from first rep of reference video):")
print("-" * 50)
for feat_idx in core_indices:
    feat_name = ANGLE_NAMES[feat_idx]
    template_signal = template_rep[:, feat_idx]
    print(f"  {feat_name:15} min={template_signal.min():6.1f}°  max={template_signal.max():6.1f}°  range={template_signal.ptp():5.1f}°")

# User reps statistics  
print("\n📊 USER REPS:")
for rep_idx, user_rep in enumerate(user_reps_data):
    print(f"\n  Rep {rep_idx + 1}:")
    print("-" * 50)
    for feat_idx in core_indices:
        feat_name = ANGLE_NAMES[feat_idx]
        user_signal = user_rep[:, feat_idx]
        template_signal = template_rep[:, feat_idx]
        
        # Resample to compare
        user_resampled = resample_to_length(user_signal, TEMPLATE_LENGTH)
        
        diff_at_bottom = template_signal.min() - user_resampled.min()
        
        print(f"    {feat_name:15} min={user_signal.min():6.1f}°  max={user_signal.max():6.1f}°  "
              f"range={user_signal.ptp():5.1f}°  (diff from template min: {diff_at_bottom:+.1f}°)")

# Check if the knee angles show different squat depths
print("\n\n📊 SQUAT DEPTH ANALYSIS (lower knee angle = deeper squat):")
print("-" * 60)
print(f"  Template knee angle at bottom: {template_rep[:, 2].min():.1f}° (left), {template_rep[:, 3].min():.1f}° (right)")
for rep_idx, user_rep in enumerate(user_reps_data):
    left_min = user_rep[:, 2].min()
    right_min = user_rep[:, 3].min()
    avg_min = (left_min + right_min) / 2
    template_avg = (template_rep[:, 2].min() + template_rep[:, 3].min()) / 2
    depth_diff = avg_min - template_avg
    
    if depth_diff > 10:
        depth_note = "SHALLOWER than template"
    elif depth_diff < -10:
        depth_note = "DEEPER than template"
    else:
        depth_note = "similar depth to template"
    
    print(f"  User Rep {rep_idx+1} knee angle at bottom: {left_min:.1f}° (left), {right_min:.1f}° (right) -- {depth_note} ({depth_diff:+.1f}°)")

In [ ]:
# DIAGNOSTIC: Check if videos might be swapped
# Compare the two video files to see which one looks more "professional"

print("="*70)
print("DIAGNOSTIC: Video Comparison - Which is the Pro?")
print("="*70)

print(f"\n📁 Reference file (currently treated as PRO): {REF_NAME}")
print(f"📁 User file (currently treated as USER): {USER_NAME}")

print("\n📊 Reference video statistics:")
print(f"   Total frames: {ref_landmarks.shape[0]}")
print(f"   Reps detected: {len(ref_reps)}")
avg_ref_depth = np.mean([ref_smooth[s:e+1, 2].min() for s, e in ref_reps])
print(f"   Average squat depth (left knee): {avg_ref_depth:.1f}°")

print("\n📊 User video statistics:")
print(f"   Total frames: {user_landmarks.shape[0]}")
print(f"   Reps detected: {len(user_reps)}")
avg_user_depth = np.mean([user_smooth[s:e+1, 2].min() for s, e in user_reps])
print(f"   Average squat depth (left knee): {avg_user_depth:.1f}°")

# Movement quality metrics (smoothness)
ref_velocity_var = np.var(np.diff(ref_smooth[:, 2]))
user_velocity_var = np.var(np.diff(user_smooth[:, 2]))

print("\n📊 Movement smoothness (lower = smoother):")
print(f"   Reference: {ref_velocity_var:.4f}")
print(f"   User:      {user_velocity_var:.4f}")

if ref_velocity_var < user_velocity_var:
    print("   -> Reference video has smoother movement (more likely to be the pro)")
else:
    print("   -> User video has smoother movement (videos might be swapped!)")

print("\n" + "="*70)
print("⚠️  IMPORTANT: Verify that the file assignments are correct!")
print(f"   - '{REF_NAME}' = narrower stance OR narrow camera angle?")
print(f"   - '{USER_NAME}' = wider stance OR wider camera angle?")
print("   - Which video shows the professional athlete?")
print("="*70)

In [ ]:
os.makedirs("../templates", exist_ok=True)

# Template file name includes the reference video name
template_path = f"../templates/{REF_NAME}_template.npz"

template_data = {
    'template': template_rep,
    'feature_names': ANGLE_NAMES,
    'core_features': core_features,
    'template_length': template_rep.shape[0],
    'exercise': 'squat',
    'source_video': REF_NAME,
    'creation_method': template_info['method'],
    'n_reps_available': template_info['n_reps_available'],
}

if template_info['method'] == 'average':
    template_data['n_reps_averaged'] = template_info.get('n_reps_averaged', 0)
elif template_info['method'] in ['first', 'best']:
    template_data['rep_used'] = template_info.get('rep_used', 1)
if 'rep_scores' in template_info:
    template_data['rep_scores'] = template_info['rep_scores']

np.savez(template_path, **template_data)
print(f"Template saved to {template_path}")
print(f"  - Source video: {REF_NAME}")
print(f"  - Template shape: {template_rep.shape}")
print(f"  - Creation method: {template_info['method']}")
if template_info['method'] == 'average':
    print(f"  - Averaged from {template_info.get('n_reps_averaged', 'N/A')} pro reps")
else:
    print(f"  - Used rep #{template_info.get('rep_used', 'N/A')}")

### Production Usage Example

How to use the saved template in your FastAPI backend:

In [ ]:
# Example: How to use in production
print(f"""
# In your FastAPI backend:

import numpy as np

# Load template once at startup
template_file = np.load("templates/{REF_NAME}_template.npz", allow_pickle=True)
SQUAT_TEMPLATE = template_file['template']
FEATURE_NAMES = list(template_file['feature_names'])
CORE_FEATURES = list(template_file['core_features'])

def analyze_user_video(landmarks_path: str) -> dict:
    landmarks = np.load(landmarks_path)
    angles = compute_angle_features_2d(landmarks)
    smooth = smooth_angles(angles)
    
    reps, _ = find_rep_boundaries(smooth, exercise_type='heavy_squat')
    rep_data = extract_rep_angles(smooth, reps)
    
    results = []
    for i, rep in enumerate(rep_data):
        result = compare_rep_to_template(rep, SQUAT_TEMPLATE, FEATURE_NAMES, CORE_FEATURES)
        feedback = generate_feedback(result, i + 1)
        results.append({{
            'rep_number': i + 1,
            'core_similarity': result['core_similarity'],
            'feedback': feedback
        }})
    
    return {{
        'n_reps': len(results),
        'average_score': np.mean([r['core_similarity'] for r in results]),
        'reps': results
    }}
""")